<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🔥 Chapter 15 — Deep Learning with Keras and TensorFlow</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 90 mins | Level: Advanced</p>
<p style='margin:4px 0 0;font-size:13px;opacity:0.85'>💡 Tip: Enable GPU runtime in Colab → Runtime → Change runtime type → T4 GPU</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Build neural networks using the Keras Sequential API
- Add Dense, Dropout, and BatchNormalization layers
- Compile with appropriate loss, optimiser, and metrics
- Train with fit(), plot learning curves, and use callbacks
- Evaluate and prevent overfitting with dropout and early stopping

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Dataset: MNIST Handwritten Digits
# ─────────────────────────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalise to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# Flatten 28x28 images to 784-dim vectors
X_train_flat = X_train.reshape(-1, 784)
X_test_flat  = X_test.reshape(-1, 784)

print(f'Train: {X_train_flat.shape}, Test: {X_test_flat.shape}')
print(f'Classes: {np.unique(y_train)}')

## 📖 Section A — Keras Sequential API

```python
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')  # 10 output classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

history = model.fit(X_train, y_train, epochs=10,
                    batch_size=128, validation_split=0.1, verbose=1)
```

> 💡 **Key Rule:** Use `sparse_categorical_crossentropy` when labels are integers (0,1,2...). Use `categorical_crossentropy` when labels are one-hot encoded. `Adam` is the default optimiser — it adapts learning rates automatically and usually outperforms vanilla SGD.

---
## 🟢 Exercise 15.1 — Build and Train MLP on MNIST *(Level 1 · Est. 15 min)*

> Build an MLP: 784 → 128 → 64 → 10 (softmax). Train 10 epochs. Plot accuracy and loss curves.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 15.1: Train MLP on MNIST
# ─────────────────────────────────────────────────────────────

# YOUR CODE HERE — build model with Sequential API
model = None

# YOUR CODE HERE — compile and train
history = None

# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# YOUR CODE HERE — plot accuracy and loss

test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')
assert test_acc > 0.95
print('✅ MNIST classifier trained!')

In [ ]:
# @title ✅ Solution — Exercise 15.1 (click to expand)

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

history = model.fit(X_train_flat, y_train,
                    epochs=10, batch_size=128,
                    validation_split=0.1, verbose=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train Acc')
axes[0].plot(history.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Accuracy'); axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss'); axes[1].legend()

plt.tight_layout()
plt.show()

test_loss, test_acc = model.evaluate(X_test_flat, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')

---
## 🟡 Exercise 15.2 — Dropout and Early Stopping *(Level 2 · Est. 20 min)*

> Build a deeper network with Dropout. Add EarlyStopping callback (patience=5). Compare to baseline: does Dropout improve generalisation?

In [ ]:
# @title ✅ Solution — Exercise 15.2 (click to expand)

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model_dropout = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_dropout.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

history_d = model_dropout.fit(X_train_flat, y_train,
                               epochs=50, batch_size=128,
                               validation_split=0.1,
                               callbacks=[early_stop],
                               verbose=0)

_, base_acc    = model.evaluate(X_test_flat, y_test, verbose=0)
_, dropout_acc = model_dropout.evaluate(X_test_flat, y_test, verbose=0)

print(f'Baseline test accuracy: {base_acc:.4f}')
print(f'Dropout test accuracy:  {dropout_acc:.4f}')
print(f'Stopped at epoch: {len(history_d.history["loss"])}')

plt.figure(figsize=(8, 4))
plt.plot(history_d.history['val_loss'], label='Dropout Val Loss')
plt.plot(history.history['val_loss'],   label='Baseline Val Loss')
plt.title('Validation Loss: Dropout vs Baseline')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.show()
print('✅ Dropout regularises by randomly zeroing neurons during training — reduces overfitting.')

---
## 🔴 Exercise 15.3 — Visualise Predictions and Errors *(Level 3 · Est. 20 min)*

> Show 10 correctly classified and 10 misclassified MNIST images. Plot a confusion matrix of the final model predictions on the test set.

In [ ]:
# @title ✅ Solution — Exercise 15.3 (click to expand)
import seaborn as sns
from sklearn.metrics import confusion_matrix

y_pred = np.argmax(model_dropout.predict(X_test_flat, verbose=0), axis=1)

correct = np.where(y_pred == y_test)[0]
wrong   = np.where(y_pred != y_test)[0]

fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i, idx in enumerate(correct[:10]):
    axes[0, i].imshow(X_test[idx], cmap='gray')
    axes[0, i].set_title(f'✓{y_test[idx]}', fontsize=8)
    axes[0, i].axis('off')
for i, idx in enumerate(wrong[:10]):
    axes[1, i].imshow(X_test[idx], cmap='gray')
    axes[1, i].set_title(f'T:{y_test[idx]} P:{y_pred[idx]}', fontsize=8, color='red')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Correct', fontsize=10)
axes[1, 0].set_ylabel('Errors', fontsize=10)
plt.suptitle('MNIST Predictions: Correct (top) vs Errors (bottom)', fontsize=12)
plt.tight_layout()
plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix — MNIST Test Set')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.show()
print('✅ Observe: most errors occur between visually similar digits (1/7, 4/9, 3/8).')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What does Dropout do and why does it work?</strong></summary>

**Answer:** Dropout randomly sets a fraction of neurons to 0 during each training step. This forces the network not to rely on any single neuron — it must learn redundant representations. At inference time, all neurons are active but outputs are scaled by (1 - dropout_rate). Why it works: it effectively trains an ensemble of sub-networks (2^n possibilities with n neurons) and averages their predictions. It's equivalent to L2 regularisation in some settings.
</details>

<details>
<summary><strong>Q2: What is the Adam optimiser?</strong></summary>

**Answer:** Adam (Adaptive Moment Estimation) combines two gradient descent improvements: Momentum (keeps a running average of gradients — accelerates in consistent directions) and RMSProp (adapts learning rates per parameter based on gradient magnitude). The result: each parameter gets its own adaptive learning rate. Adam is robust to learning rate choice and works well out of the box. Default learning_rate=0.001 works for most problems.
</details>

<details>
<summary><strong>Q3: What is batch size and how does it affect training?</strong></summary>

**Answer:** Batch size is the number of samples used to compute one gradient update. Large batches (512, 1024): more stable gradients, faster per-epoch with GPU, but may generalise worse (sharp minima). Small batches (16, 32): noisier gradients, more updates per epoch, often generalise better (flat minima). Typical default: 32–128. Rule: if training is slow, increase batch size. If model is overfitting or underfitting, adjust other hyperparameters first.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 15 Complete!</h3>
<ul>
<li>Keras Sequential API: Dense, Dropout, activation functions</li>
<li>Compile with Adam, sparse_categorical_crossentropy</li>
<li>Training with fit(), learning curves, EarlyStopping</li>
<li>Visualisation of errors and confusion matrix</li>
</ul>
<p><strong>Next:</strong> Chapter 16 — End-to-End Machine Learning Projects</p>
</div>